## PLN 2026 - Trabalho da disciplina
Baseado no exemplo da Aula 6 (Prof. Maiko Spiess)

---

## Agrupamento de notícias de futebol do Lance!

Este notebook aplica, sobre a base coletada pelo nosso projeto ([github.com/pkalbuquerque/pln](https://github.com/pkalbuquerque/pln)), a mesma sequência de técnicas de PLN demonstrada em aula com o dataset sintético de animais.

A base é composta por notícias de futebol brasileiro publicadas pelo [Lance!](https://www.lance.com.br/), coletadas pelo script `coletar_lance.py` e pré-processadas pelo script `preprocessar.py` do repositório do projeto.

As etapas realizadas aqui são:

1.  **Carregamento dos dados**: importação da base `outputs/noticias_lance_processadas.csv`, já coletada e pré-processada pelo nosso pipeline.
2.  **Preparação dos textos**: nesta base, a tokenização, normalização, remoção de *stopwords*, *stemming* e lematização já foram feitas pelo próprio pipeline do projeto (`preprocessar.py`). Aqui apenas reconstruímos essas colunas e conferimos a base.
3.  **Representação numérica**: transformação das notícias em vetores numéricos usando **Bag of Words (BoW)** e **TF-IDF**.
4.  **Cálculo de similaridade**: similaridade do cosseno entre as notícias, para as representações BoW e TF-IDF.
5.  **Agrupamento (Clustering)**: aplicação do algoritmo K-Means, com escolha de *k* orientada tanto pelas seções/editorias da base quanto pelo método do cotovelo e pelo coeficiente de silhueta.
6.  **Análise de clusters**: visualização com PCA em 2D e 3D, interpretação por termos mais representativos, coesão interna e distância entre clusters.


## Carregando a base de dados

Vamos trabalhar com a base processada gerada pelo pipeline do projeto: `outputs/noticias_lance_processadas.csv`.

Segundo o `DICIONARIO_DADOS.md` do repositório, essa base possui, entre outras, as colunas:

- `id`: identificador determinístico calculado a partir da URL da notícia;
- `titulo`: título da matéria;
- `secao`: editoria ou clube associado à matéria (equivalente à `categoria` do exemplo da aula);
- `publicado_em`: data/hora de publicação (UTC);
- `url`: endereço canônico da matéria;
- `texto_normalizado`: texto em minúsculas, sem URLs, e-mails, emojis e com Unicode normalizado;
- `tokens_sem_stopwords`: tokens após remoção de stopwords em português (serializados como termos separados por espaço);
- `stems`: radicais gerados pelo algoritmo Snowball para português;
- `lemas`: formas canônicas estimadas pela biblioteca `simplemma`;
- `quantidade_tokens` / `quantidade_tokens_limpos`: contagens de tokens antes/depois da limpeza.

Diferente do exemplo da aula (que carregava uma coluna única de descrição crua), aqui a etapa de preparação textual **já foi feita pelo pipeline de coleta do projeto**. Vamos aproveitar isso.

Rode a célula abaixo: se estiver no Google Colab, ela vai pedir para você selecionar o arquivo `noticias_lance_processadas.csv` (gerado ao rodar `python pipeline.py` ou `python preprocessar.py` localmente); se estiver rodando localmente/Jupyter, ela tenta ler o arquivo direto da pasta `outputs/` do projeto.

In [ ]:
import pandas as pd

# Tenta detectar se estamos no Google Colab; caso contrário, lê o arquivo
# direto da pasta outputs/ do projeto (rodando localmente, ex.: jupyter notebook)
try:
    from google.colab import files

    print("Selecione o arquivo outputs/noticias_lance_processadas.csv")
    uploaded = files.upload()
    nome_arquivo = next(iter(uploaded))

except ImportError:
    nome_arquivo = "outputs/noticias_lance_processadas.csv"
    print(f"Ambiente local detectado. Lendo diretamente: {nome_arquivo}")

# Carregar os dados
df = pd.read_csv(nome_arquivo)

# Visualizar as primeiras linhas
df.head()

Selecione o arquivo outputs/noticias_lance_processadas.csv


Saving noticias_lance_processadas.csv to noticias_lance_processadas.csv


,id,titulo,secao,publicado_em,url,texto_normalizado,tokens_sem_stopwords,stems,lemas,quantidade_tokens,quantidade_tokens_limpos
0,879e466ddd05b6b8,Jardim define escalação do Flamengo contra o D...,Flamengo,2026-09-17T23:05:41Z,https://www.lance.com.br/flamengo/libertadores...,o flamengo está pronto para a partida contra o...,flamengo pronto partida independiente del vall...,flameng pront part independient del vall jog v...,flamengo pronto partida independiente del vall...,357,215
1,cd87266b51a9465f,'Fim do futebol': clubes da Série A se manifes...,Futebol Nacional,2026-09-17T22:33:42Z,https://www.lance.com.br/futebol-nacional/fim-...,oito clubes da série a do futebol brasileiro p...,oito clubes série futebol brasileiro publicara...,oit club séri futebol brasileir public noit qu...,oito clube série futebol brasileiro publicar n...,720,416
2,a9b970813b1cd3c9,Números de Viña em comparação com outros later...,Cruzeiro,2026-09-17T22:05:17Z,https://www.lance.com.br/cruzeiro/numeros-vina...,"para o duelo do próximo domingo (20), contra o...",duelo próximo domingo vitória técnico artur jo...,duel próxim doming vitór técnic artur jorg ter...,duelo próximo domingo vitória técnico artur jo...,285,159
3,ff7d69075e3dcf7d,Corinthians abre conversas para renovar com meia,Corinthians,2026-09-17T21:56:00Z,https://www.lance.com.br/corinthians/corinthia...,o corinthians negocia a renovação de contrato ...,corinthians negocia renovação contrato meia an...,corinthians negoc renov contrat mei andré carr...,corinthians negociar renovação contrato meio a...,272,153
4,b1733bf22727e743,Corinthians blinda Memphis após pênalti perdid...,Corinthians,2026-09-17T21:26:28Z,https://www.lance.com.br/corinthians/corinthia...,marcelo paz saiu em defesa de memphis depay ap...,marcelo paz saiu defesa memphis depay atacante...,marcel paz saiu defes memph depay atac desperd...,marcelo paz sair defesa memphis depay atacante...,606,310


## Baixando as tabelas geradas

Para facilitar a entrega do trabalho, cada tabela importante produzida ao longo deste notebook é salva automaticamente em CSV na pasta `tabelas_exportadas/`.

- No **Google Colab**, o download para o seu computador é disparado automaticamente assim que a célula termina de rodar.
- Rodando **localmente** (Jupyter), os arquivos ficam disponíveis na pasta `tabelas_exportadas/`, dentro da pasta onde está este notebook.

In [ ]:
import os

# Pasta onde todas as tabelas geradas neste notebook serão salvas
PASTA_TABELAS = "tabelas_exportadas"
os.makedirs(PASTA_TABELAS, exist_ok=True)


def baixar_tabela(tabela, nome_arquivo, index=False):
    """Salva uma tabela (DataFrame) como CSV e, se estivermos no Google
    Colab, dispara o download automático do arquivo para o computador."""

    caminho = os.path.join(PASTA_TABELAS, nome_arquivo)
    tabela.to_csv(caminho, index=index, encoding="utf-8")

    print(f"Tabela salva em: {caminho}")

    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        # Fora do Colab (ex.: Jupyter local), o arquivo já fica
        # disponível na pasta 'tabelas_exportadas/' do projeto.
        pass

## Preparação dos textos

No exemplo da aula, a tokenização, a normalização e a remoção de *stopwords* eram feitas dentro do próprio notebook, pois a base de animais chegava como texto puro.

No nosso projeto, essas etapas já acontecem em `preprocessar.py`, na coleta:

1. normalização Unicode NFC e conversão para minúsculas;
2. remoção de URLs, e-mails e emojis;
3. remoção de números e pontuação durante a tokenização;
4. tokenização por palavras;
5. remoção de stopwords em português;
6. *stemming* com o algoritmo Snowball para português;
7. lematização em português com `simplemma`.

Por isso, aqui não repetimos esse trabalho — apenas reconstruímos as colunas de listas (que no CSV vêm como texto com termos separados por espaço) e conferimos a base antes de seguir para as representações numéricas.

In [ ]:
# As colunas de tokens/stems/lemas vêm no CSV como termos separados por
# espaço (ver preprocessar.py); aqui reconstruímos as listas em Python.

def para_lista(valor):
    if pd.isna(valor):
        return []
    return str(valor).split()


df["tokens_sem_stopwords"] = df["tokens_sem_stopwords"].apply(para_lista)
df["stems"] = df["stems"].apply(para_lista)
df["lemas"] = df["lemas"].apply(para_lista)

# Documentos sem seção identificada recebem um rótulo explícito
df["secao"] = df["secao"].fillna("Sem seção")

# Estatísticas rápidas da base
n_documentos = len(df)
contagem_secoes = df["secao"].value_counts()

print(f"Total de notícias: {n_documentos}")
print(f"Seções/editorias distintas: {contagem_secoes.shape[0]}")
print(f"Média de tokens por notícia (sem stopwords): {df['quantidade_tokens_limpos'].mean():.1f}")
print()
print("Notícias por seção:")
print(contagem_secoes)

df[
    [
        "titulo",
        "secao",
        "tokens_sem_stopwords",
        "stems",
        "lemas",
        "quantidade_tokens",
        "quantidade_tokens_limpos"
    ]
].head()

Total de notícias: 500
Seções/editorias distintas: 13
Média de tokens por notícia (sem stopwords): 227.4

Notícias por seção:
secao
Futebol Nacional    65
Corinthians         55
Vasco               54
Palmeiras           54
Santos              50
Flamengo            45
São Paulo           44
Cruzeiro            37
Fluminense          36
Atlético Mineiro    28
Botafogo            25
Grêmio               6
Copa do Brasil       1
Name: count, dtype: int64


,titulo,secao,tokens_sem_stopwords,stems,lemas,quantidade_tokens,quantidade_tokens_limpos
0,Jardim define escalação do Flamengo contra o D...,Flamengo,"[flamengo, pronto, partida, independiente, del...","[flameng, pront, part, independient, del, vall...","[flamengo, pronto, partida, independiente, del...",357,215
1,'Fim do futebol': clubes da Série A se manifes...,Futebol Nacional,"[oito, clubes, série, futebol, brasileiro, pub...","[oit, club, séri, futebol, brasileir, public, ...","[oito, clube, série, futebol, brasileiro, publ...",720,416
2,Números de Viña em comparação com outros later...,Cruzeiro,"[duelo, próximo, domingo, vitória, técnico, ar...","[duel, próxim, doming, vitór, técnic, artur, j...","[duelo, próximo, domingo, vitória, técnico, ar...",285,159
3,Corinthians abre conversas para renovar com meia,Corinthians,"[corinthians, negocia, renovação, contrato, me...","[corinthians, negoc, renov, contrat, mei, andr...","[corinthians, negociar, renovação, contrato, m...",272,153
4,Corinthians blinda Memphis após pênalti perdid...,Corinthians,"[marcelo, paz, saiu, defesa, memphis, depay, a...","[marcel, paz, saiu, defes, memph, depay, atac,...","[marcelo, paz, sair, defesa, memphis, depay, a...",606,310


## Representação 1 — Bag of Words (BoW)

O **Bag of Words (BoW)** representa cada documento pela frequência das palavras que aparecem nele. A ordem das palavras é ignorada.

Como nosso pipeline já entrega três versões dos tokens (`tokens_sem_stopwords`, `stems` e `lemas`), escolhemos uma coluna-base para as representações numéricas. Usaremos `lemas`, pois a lematização reduz variações morfológicas (plural, conjugação, etc.) mantendo palavras reconhecíveis — o que costuma ajudar o K-Means a agrupar notícias sobre o mesmo assunto mesmo quando o texto usa flexões diferentes.

> Quer comparar com outra representação? Basta trocar o valor de `coluna_base` abaixo por `"tokens_sem_stopwords"` ou `"stems"` e rodar novamente as células seguintes.

O campo `id` é mantido para relacionar cada representação à base original, assim como no exemplo da aula.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Coluna de tokens usada como base para as representações numéricas
coluna_base = "lemas"

# Criar o modelo Bag of Words
# Como os textos já estão tokenizados, usamos diretamente as listas de tokens
bow_vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens
)

# Criar a matriz Bag of Words
bow_matrix = bow_vectorizer.fit_transform(
    df[coluna_base]
)

# Recuperar o vocabulário
bow_palavras = bow_vectorizer.get_feature_names_out()

print(f"Tamanho do vocabulário (BoW): {len(bow_palavras)} termos")

# Transformar a matriz em DataFrame
df_bow = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_palavras
)

# Adicionar o ID para conectar com a base original
df_bow.insert(0, "id", df["id"].values)

# Salvar/baixar a tabela completa
baixar_tabela(df_bow, "01_bow.csv")

# Visualizar (apenas as primeiras linhas — a tabela completa foi salva acima)
df_bow.head()

Tamanho do vocabulário (BoW): 6354 termos
Tabela salva em: tabelas_exportadas/01_bow.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id,João,Portugal,abafador,abafar,abaixar,abaixo,abalar,abate,abater,...,índio,íntegro,óbvio,ódio,órgão,ótimo,ônibus,último,único,útil
0,879e466ddd05b6b8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,cd87266b51a9465f,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0
2,a9b970813b1cd3c9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
3,ff7d69075e3dcf7d,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,b1733bf22727e743,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


## Representação 2 — TF-IDF

O **TF-IDF** também representa cada documento pelas palavras presentes nele, mas atribui pesos diferentes:

- **TF (*Term Frequency*)**: quanto uma palavra aparece em determinado documento;
- **IDF (*Inverse Document Frequency*)**: reduz o peso de palavras que aparecem em muitas notícias (como termos genéricos de futebol) e aumenta o peso de palavras mais específicas (nomes de clubes, jogadores, competições).

Em notícias esportivas, isso é especialmente útil: palavras como "jogo", "time" ou "partida" tendem a aparecer em quase todas as matérias e, por isso, recebem peso baixo, enquanto nomes próprios e termos específicos de cada notícia recebem peso mais alto.

Assim como no BoW, mantemos o `id` para relacionar essa representação à base original.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criar o modelo TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

# Criar a matriz TF-IDF
tfidf_matrix = tfidf_vectorizer.fit_transform(
    df[coluna_base]
)

# Recuperar o vocabulário
tfidf_palavras = tfidf_vectorizer.get_feature_names_out()

# Transformar a matriz em DataFrame
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_palavras
)

# Adicionar o ID para conectar com a base original
df_tfidf.insert(0, "id", df["id"].values)

# Salvar/baixar a tabela completa
baixar_tabela(df_tfidf, "02_tfidf.csv")

# Visualizar (apenas as primeiras linhas — a tabela completa foi salva acima)
df_tfidf.head(20)

Tabela salva em: tabelas_exportadas/02_tfidf.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id,João,Portugal,abafador,abafar,abaixar,abaixo,abalar,abate,abater,...,índio,íntegro,óbvio,ódio,órgão,ótimo,ônibus,último,único,útil
0,879e466ddd05b6b8,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.026788,0.000000,0.0
1,cd87266b51a9465f,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.044781,0.0,0.0,0.0,0.0,0.0,0.013694,0.000000,0.0
2,a9b970813b1cd3c9,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.029029,0.054470,0.0
3,ff7d69075e3dcf7d,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.033476,0.000000,0.0
4,b1733bf22727e743,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.036780,0.0
5,6a26a92814a22f7c,0.072632,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.074952,0.000000,0.0
6,990314c6d34376c9,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
7,1b981fcf7a99bedb,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.029915,0.000000,0.0
8,850f2b82f5f43d55,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.150041,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0
9,be2d4192994f22c1,0.000000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.020282,0.000000,0.0


## Similaridade entre as notícias

Com cada notícia transformada em um vetor numérico, podemos comparar os documentos entre si usando a **similaridade do cosseno**:

- valores próximos de **1** indicam textos muito semelhantes;
- valores próximos de **0** indicam pouca semelhança.

Calculamos duas matrizes de similaridade: uma a partir do **Bag of Words** e outra a partir do **TF-IDF**. Cada posição da matriz representa a similaridade entre duas notícias.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Rótulo curto para identificar cada notícia nos eixos/tooltips dos gráficos
def truncar(texto, tamanho=60):
    texto = str(texto)
    return texto if len(texto) <= tamanho else texto[:tamanho - 1] + "…"


rotulos = df["titulo"].apply(truncar)

# -------------------------------------------------
# Similaridade a partir do Bag of Words
# -------------------------------------------------

similaridade_bow = cosine_similarity(
    df_bow.drop(columns="id")
)

df_sim_bow = pd.DataFrame(
    similaridade_bow,
    index=rotulos,
    columns=rotulos
)


# -------------------------------------------------
# Similaridade a partir do TF-IDF
# -------------------------------------------------

similaridade_tfidf = cosine_similarity(
    df_tfidf.drop(columns="id")
)

df_sim_tfidf = pd.DataFrame(
    similaridade_tfidf,
    index=rotulos,
    columns=rotulos
)


# Salvar/baixar as tabelas completas
baixar_tabela(df_sim_bow, "03_similaridade_bow.csv", index=True)
baixar_tabela(df_sim_tfidf, "04_similaridade_tfidf.csv", index=True)

# Visualizar uma das matrizes (apenas as primeiras linhas)
df_sim_bow.head()

Tabela salva em: tabelas_exportadas/03_similaridade_bow.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tabela salva em: tabelas_exportadas/04_similaridade_tfidf.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

titulo,Jardim define escalação do Flamengo contra o Del Valle,'Fim do futebol': clubes da Série A se manifestam contra pr…,Números de Viña em comparação com outros laterais do Cruzei…,Corinthians abre conversas para renovar com meia,Corinthians blinda Memphis após pênalti perdido na Libertad…,Cruzeiro pode ter retorno importante contra o Vitória,Neymar se pronuncia após queda do Santos na Sul-Americana,Corinthians decide futuro de Diniz após queda na Libertador…,"Goleiro do mês! Confira números de Otávio, do Cruzeiro",Provocações de Deyverson; relembre momentos do atacante,...,São Paulo perde Calleri para decisão contra o Boca,Dorival admite frustração com derrota do São Paulo para o B…,Luciano revela se há possível lesão após partida contra o B…,Domingos Duarte manda recado após derrota do São Paulo para…,São Paulo perde para o Boca na Bombonera e decide vaga no M…,Veja gol em Boca Juniors x São Paulo: Merentiel marca,Pedro Emanuel valoriza resultado do Vasco e evita comentar …,Marcão destaca empenho dos jogadores do Fluminense: 'Compro…,Ziyech desembarca no Rio para assinar com o Botafogo,Marino lamenta chances perdidas em empate do Vasco na Sula
titulo,,,,,,,,,,,,,,,,,,,,,
Jardim define escalação do Flamengo contra o Del Valle,1.000000,0.070840,0.131601,0.178935,0.141770,0.166010,0.130702,0.198817,0.107229,0.135182,...,0.165355,0.134566,0.190851,0.130599,0.212674,0.173894,0.229575,0.251058,0.104803,0.167930
'Fim do futebol': clubes da Série A se manifestam contra pr…,0.070840,1.000000,0.072136,0.127457,0.126195,0.112581,0.129250,0.153837,0.162033,0.157460,...,0.089424,0.078371,0.110640,0.046941,0.062285,0.065189,0.118424,0.086277,0.087759,0.076558
Números de Viña em comparação com outros laterais do Cruzei…,0.131601,0.072136,1.000000,0.138826,0.168962,0.287972,0.086918,0.153287,0.128162,0.091770,...,0.153948,0.243294,0.171954,0.145907,0.123752,0.097697,0.233775,0.184196,0.121966,0.104408
Corinthians abre conversas para renovar com meia,0.178935,0.127457,0.138826,1.000000,0.287772,0.150776,0.206193,0.365927,0.143607,0.202021,...,0.166801,0.200722,0.217907,0.170108,0.164602,0.138662,0.237677,0.224084,0.191456,0.175872
Corinthians blinda Memphis após pênalti perdido na Libertad…,0.141770,0.126195,0.168962,0.287772,1.000000,0.084451,0.120821,0.340774,0.120555,0.250215,...,0.191069,0.277664,0.211045,0.179078,0.167108,0.152434,0.261959,0.291150,0.091151,0.194930


## Visualizando as matrizes de similaridade

Assim como no exemplo da aula, cada matriz de similaridade pode ser vista como um **mapa de calor (*heatmap*)**: tons mais escuros indicam maior similaridade entre duas notícias, e a diagonal principal vale sempre 1.

Como nossa base tem muito mais texto por documento do que as descrições curtas de animais, o *tooltip* mostra o título, a seção e um trecho do texto normalizado de cada notícia, em vez da descrição completa.

**Atenção ao tamanho da base:** um heatmap com *tooltip* detalhado cresce em O(n²) — com poucas dezenas de notícias ele é leve, mas com centenas de notícias ele fica pesadíssimo (um notebook de teste com 500 notícias chegou a mais de 250 MB só com esse gráfico!). Por isso, se a base tiver mais que `MAX_DOCS_HEATMAP` notícias, a célula abaixo desenha o heatmap **completo, mas sem o tooltip detalhado por par** (mantém a visão geral da estrutura de similaridade) e, além disso, plota um segundo heatmap com **tooltip completo apenas para uma amostra aleatória** de notícias, para você conseguir explorar exemplos concretos sem travar o notebook.

In [ ]:
import random
import numpy as np
import plotly.graph_objects as go

titulos = df["titulo"].tolist()
secoes = df["secao"].tolist()
trechos = df["texto_normalizado"].apply(lambda t: truncar(t, 120)).tolist()

# Acima desse número de notícias, o tooltip par a par fica pesado demais
# (cresce em O(n²)); nesse caso, usamos uma amostra para o heatmap detalhado.
MAX_DOCS_HEATMAP = 60


def criar_heatmap_simples(matriz, titulo_grafico):
    # Heatmap leve, sem tooltip detalhado por par — bom para ver a
    # estrutura geral de similaridade em bases grandes.
    fig = go.Figure(
        data=go.Heatmap(
            z=matriz,
            colorscale="Blues",
            zmin=0,
            zmax=1,
            hovertemplate="Similaridade: %{z:.3f}<extra></extra>"
        )
    )

    fig.update_layout(
        title=titulo_grafico,
        xaxis_title="Notícia (índice)",
        yaxis_title="Notícia (índice)",
        width=800,
        height=750
    )

    fig.show()


def criar_heatmap_detalhado(matriz, indices, titulo_grafico):
    # Heatmap com tooltip completo (título, seção e trecho do texto),
    # usado apenas para um subconjunto de notícias.
    sub_matriz = matriz[np.ix_(indices, indices)]
    sub_rotulos = [rotulos[i] for i in indices]
    sub_titulos = [titulos[i] for i in indices]
    sub_secoes = [secoes[i] for i in indices]
    sub_trechos = [trechos[i] for i in indices]

    customdata = [
        [
            [sub_titulos[i], sub_secoes[i], sub_trechos[i],
             sub_titulos[j], sub_secoes[j], sub_trechos[j]]
            for j in range(len(indices))
        ]
        for i in range(len(indices))
    ]

    fig = go.Figure(
        data=go.Heatmap(
            z=sub_matriz,
            x=sub_rotulos,
            y=sub_rotulos,
            colorscale="Blues",
            zmin=0,
            zmax=1,
            customdata=customdata,

            hovertemplate=
                "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                "%{customdata[2]}<br><br>"
                "<b>%{customdata[3]}</b> (%{customdata[4]})<br>"
                "%{customdata[5]}<br><br>"
                "<b>Similaridade:</b> %{z:.3f}"
                "<extra></extra>"
        )
    )

    fig.update_layout(
        title=titulo_grafico,
        xaxis_title="Notícia",
        yaxis_title="Notícia",
        width=900,
        height=850
    )

    fig.show()


if n_documentos > MAX_DOCS_HEATMAP:
    print(
        f"A base tem {n_documentos} notícias (> {MAX_DOCS_HEATMAP}): "
        f"mostrando o heatmap completo de forma simplificada e um heatmap "
        f"detalhado apenas para uma amostra de {MAX_DOCS_HEATMAP} notícias."
    )
    random.seed(42)
    indices_amostra = sorted(random.sample(range(n_documentos), MAX_DOCS_HEATMAP))

    criar_heatmap_simples(similaridade_tfidf, "Similaridade (TF-IDF) — base completa")
    criar_heatmap_detalhado(similaridade_tfidf, indices_amostra, "Similaridade (TF-IDF) — amostra detalhada")

else:
    criar_heatmap_detalhado(similaridade_bow, list(range(n_documentos)), "Similaridade entre notícias — Bag of Words")
    criar_heatmap_detalhado(similaridade_tfidf, list(range(n_documentos)), "Similaridade entre notícias — TF-IDF")

# Dica: para explorar uma seção específica em detalhe, filtre antes de plotar:
# indices_secao = df.index[df["secao"] == "Vasco"].tolist()
# criar_heatmap_detalhado(similaridade_tfidf, indices_secao, "TF-IDF — Vasco")

A base tem 500 notícias (> 60): mostrando o heatmap completo de forma simplificada e um heatmap detalhado apenas para uma amostra de 60 notícias.


## Agrupando as notícias com K-Means

Vamos identificar **grupos de notícias semelhantes** automaticamente com o algoritmo **K-Means**, usando a representação TF-IDF.

Diferente do exemplo da aula — em que o número de categorias (6 animais) já era conhecido de antemão —, aqui não sabemos, a priori, qual é o número "certo" de grupos. Isso porque:

- a coluna `secao` nos dá uma pista (clubes e editorias já catalogados), mas notícias sobre o mesmo assunto podem estar espalhadas por seções diferentes (ou vice-versa);
- o número de seções distintas pode ser grande ou pequeno demais para formar clusters interpretáveis.

Por isso, vamos combinar duas fontes de informação para escolher *k*:

1. o número de seções distintas na base, como ponto de partida;
2. o **método do cotovelo** (inércia) e o **coeficiente de silhueta**, para validar estatisticamente essa escolha.

In [ ]:
# Ponto de partida para k: número de seções distintas, limitado a uma faixa
# razoável para manter os clusters interpretáveis (entre 2 e 12 grupos).
n_secoes = contagem_secoes.shape[0]
k_base = min(max(n_secoes, 2), 12)

print(f"Seções distintas na base: {n_secoes}")
print(f"k inicial sugerido a partir das seções: {k_base}")

Seções distintas na base: 13
k inicial sugerido a partir das seções: 12


### Escolha de *k*: método do cotovelo e coeficiente de silhueta

O **método do cotovelo** observa como a inércia (soma das distâncias quadráticas dos pontos aos centróides do seu cluster) diminui à medida que *k* aumenta. Ela sempre cai, mas o "cotovelo" do gráfico indica o ponto a partir do qual aumentar *k* deixa de trazer ganhos relevantes.

O **coeficiente de silhueta** mede o quão bem cada documento se encaixa no seu cluster em comparação aos demais, variando de -1 a 1 (quanto maior, melhor a separação entre os grupos).

Vamos testar uma faixa de valores de *k* ao redor de `k_base` e usar essas duas métricas para confirmar (ou ajustar) o valor final de *k*.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.graph_objects as go

# Faixa de k a testar, limitada pelo tamanho da base
k_min = 2
k_max = min(15, n_documentos - 1)

resultados_k = []

for k_teste in range(k_min, k_max + 1):
    modelo_teste = KMeans(
        n_clusters=k_teste,
        random_state=42,
        n_init=10
    )
    labels_teste = modelo_teste.fit_predict(tfidf_matrix)

    resultados_k.append({
        "k": k_teste,
        "inercia": modelo_teste.inertia_,
        "silhueta": silhouette_score(tfidf_matrix, labels_teste)
    })

df_avaliacao_k = pd.DataFrame(resultados_k)

# k que maximiza o coeficiente de silhueta
k_otimo = int(
    df_avaliacao_k.loc[df_avaliacao_k["silhueta"].idxmax(), "k"]
)

baixar_tabela(df_avaliacao_k, "05_avaliacao_k.csv")

print(f"k sugerido pelas seções da base: {k_base}")
print(f"k sugerido pela silhueta: {k_otimo}")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_avaliacao_k["k"],
    y=df_avaliacao_k["inercia"],
    mode="lines+markers",
    name="Inércia (cotovelo)"
))

fig.update_layout(
    title="Método do cotovelo — inércia por k",
    xaxis_title="k (número de clusters)",
    yaxis_title="Inércia"
)

fig.show()

fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df_avaliacao_k["k"],
    y=df_avaliacao_k["silhueta"],
    mode="lines+markers",
    name="Coeficiente de silhueta",
    line=dict(color="darkorange")
))

fig2.update_layout(
    title="Coeficiente de silhueta por k",
    xaxis_title="k (número de clusters)",
    yaxis_title="Silhueta média"
)

fig2.show()

Tabela salva em: tabelas_exportadas/05_avaliacao_k.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

k sugerido pelas seções da base: 12
k sugerido pela silhueta: 12


### Definindo o valor final de *k*

Use o gráfico do cotovelo e o de silhueta para confirmar (ou ajustar manualmente) o valor de `k` antes de rodar o K-Means final. Por padrão, adotamos o valor sugerido pela silhueta (`k_otimo`); se o "cotovelo" indicar claramente outro valor, ou se a interpretação dos clusters fizer mais sentido de negócio com outro número, altere `k` livremente na célula abaixo.

In [ ]:
# Valor final de k usado no restante do notebook.
# Altere esta linha manualmente se quiser testar outro valor.
k = k_otimo

print(f"k final escolhido: {k}")

k final escolhido: 12


In [ ]:
# Criar e ajustar o modelo final
kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(tfidf_matrix)

# Adicionar o resultado ao dataframe original
df["cluster"] = clusters

df_clusters_finais = df[
    ["id", "titulo", "secao", "cluster"]
].sort_values("cluster")

baixar_tabela(df_clusters_finais, "06_noticias_com_cluster.csv")

# Visualizar os resultados
df_clusters_finais

Tabela salva em: tabelas_exportadas/06_noticias_com_cluster.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,id,titulo,secao,cluster
470,a53af6d16768f5b8,"Jorginho, do Flamengo, está na pré-lista da se...",Flamengo,0
29,be5269684619d8dd,Fluminense encerra jejum de dois anos com conv...,Fluminense,0
468,dc1310f732165675,"Pedro e Samuel Lino, do Flamengo, são convocad...",Flamengo,0
463,c204ddf931f0c218,Cruzeiro: Cássio parabeniza Otávio por convocação,Cruzeiro,0
457,42e0ba23308baba8,"Pedro, do Flamengo, reage à convocação para Se...",Flamengo,0
...,...,...,...,...
452,6613f24598ceefad,Richarlison reage a publicação e insinua possi...,Vasco,11
471,43964ba7b7102dc4,Alvo do Vasco: como foram últimas temporadas d...,Vasco,11
35,b242350db298de22,Vasco supera meta financeira com campanhas nas...,Vasco,11
469,9bf7ebf4914f38eb,Vasco faz proposta por Richarlison e aguarda s...,Vasco,11


## Comparando clusters e seções

Os números dos clusters são identificadores arbitrários: o `cluster 0` não corresponde necessariamente à primeira seção da base.

Uma forma simples de avaliar o resultado é observar quantas notícias de cada seção foram agrupadas em cada cluster.

In [ ]:
comparacao_clusters = pd.crosstab(
    df["secao"],
    df["cluster"]
)

baixar_tabela(comparacao_clusters, "07_comparacao_clusters_secoes.csv", index=True)

comparacao_clusters

Tabela salva em: tabelas_exportadas/07_comparacao_clusters_secoes.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

cluster,0,1,2,3,4,5,6,7,8,9,10,11
secao,,,,,,,,,,,,
Atlético Mineiro,1,0,0,0,0,3,0,24,0,0,0,0
Botafogo,0,0,0,0,0,0,25,0,0,0,0,0
Copa do Brasil,0,0,0,0,0,0,0,1,0,0,0,0
Corinthians,3,40,0,0,0,12,0,0,0,0,0,0
Cruzeiro,7,0,17,0,8,4,1,0,0,0,0,0
Flamengo,4,27,0,0,0,14,0,0,0,0,0,0
Fluminense,4,0,0,0,0,5,0,1,0,26,0,0
Futebol Nacional,0,4,2,0,1,30,5,5,2,2,9,5
Grêmio,0,0,0,0,0,0,6,0,0,0,0,0


## Visualizando os clusters com PCA

Nossa representação TF-IDF tem muitas dimensões: cada termo do vocabulário é uma dimensão. Para visualizar a distribuição das notícias, usamos **PCA (Principal Component Analysis)**, uma técnica de redução de dimensionalidade.

Primeiro, reduzimos os vetores TF-IDF para **duas dimensões**. Cada ponto representa uma notícia, e sua cor indica o cluster identificado pelo K-Means.

> O PCA é usado apenas para visualização — os clusters já foram calculados com a representação TF-IDF completa.

In [ ]:
from sklearn.decomposition import PCA
import plotly.express as px

# Reduzir o TF-IDF para duas dimensões
pca_2d = PCA(n_components=2)

coordenadas_2d = pca_2d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar um DataFrame para a visualização
df_pca_2d = pd.DataFrame({
    "id": df["id"],
    "titulo": df["titulo"],
    "secao": df["secao"],
    "cluster": df["cluster"].astype(str),
    "PCA1": coordenadas_2d[:, 0],
    "PCA2": coordenadas_2d[:, 1]
})

baixar_tabela(df_pca_2d, "08_pca_2d.csv")

# Criar o gráfico
fig = px.scatter(
    df_pca_2d,
    x="PCA1",
    y="PCA2",
    color="cluster",
    hover_name="titulo",
    hover_data={
        "secao": True,
        "PCA1": False,
        "PCA2": False
    },
    title="Clusters das notícias — PCA em 2 dimensões"
)

fig.update_traces(
    marker=dict(size=10)
)

fig.update_layout(
    xaxis_title="Componente principal 1",
    yaxis_title="Componente principal 2",
    legend_title="Cluster"
)

fig.show()

Tabela salva em: tabelas_exportadas/08_pca_2d.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Visualização em três dimensões

Repetimos o procedimento mantendo **três componentes principais**. A terceira dimensão pode preservar parte da informação perdida na projeção bidimensional, e o gráfico interativo permite girar o espaço e observar os grupos sob diferentes ângulos.

In [ ]:
# Reduzir o TF-IDF para três dimensões
pca_3d = PCA(n_components=3)

coordenadas_3d = pca_3d.fit_transform(
    tfidf_matrix.toarray()
)

# Criar DataFrame para visualização
df_pca_3d = pd.DataFrame({
    "id": df["id"],
    "titulo": df["titulo"],
    "secao": df["secao"],
    "cluster": df["cluster"].astype(str),
    "PCA1": coordenadas_3d[:, 0],
    "PCA2": coordenadas_3d[:, 1],
    "PCA3": coordenadas_3d[:, 2]
})

baixar_tabela(df_pca_3d, "09_pca_3d.csv")

# Criar gráfico 3D
fig = px.scatter_3d(
    df_pca_3d,
    x="PCA1",
    y="PCA2",
    z="PCA3",
    color="cluster",
    hover_name="titulo",
    hover_data={
        "secao": True,
        "PCA1": False,
        "PCA2": False,
        "PCA3": False
    },
    title="Clusters das notícias — PCA em 3 dimensões"
)

fig.update_traces(
    marker=dict(size=5)
)

fig.update_layout(
    scene=dict(
        xaxis_title="Componente principal 1",
        yaxis_title="Componente principal 2",
        zaxis_title="Componente principal 3"
    ),
    legend_title="Cluster"
)

fig.show()

Tabela salva em: tabelas_exportadas/09_pca_3d.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
variancia_2d = round(pca_2d.explained_variance_ratio_.sum(), 3)
variancia_3d = round(pca_3d.explained_variance_ratio_.sum(), 3)

df_variancia_explicada = pd.DataFrame({
    "componentes": ["2D", "3D"],
    "variancia_explicada": [variancia_2d, variancia_3d]
})

baixar_tabela(df_variancia_explicada, "10_variancia_explicada.csv")

print("Variância explicada em 2D:", variancia_2d)
print("Variância explicada em 3D:", variancia_3d)

Tabela salva em: tabelas_exportadas/10_variancia_explicada.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Variância explicada em 2D: 0.048
Variância explicada em 3D: 0.068


## Quais termos representam cada cluster?

O K-Means cria um **centróide** para cada cluster — um vetor médio dos documentos daquele grupo. Como trabalhamos com TF-IDF, podemos inspecionar quais termos têm os maiores pesos em cada centróide.

Esses termos ajudam a interpretar semanticamente os clusters: em notícias de futebol, esperamos ver nomes de clubes, competições ou jogadores recorrentes em cada grupo.

In [ ]:
import numpy as np

# Nomes dos termos do vocabulário TF-IDF
termos = tfidf_vectorizer.get_feature_names_out()

# Centróides calculados pelo K-Means
centroides = kmeans.cluster_centers_

# Quantidade de termos que queremos mostrar
n_termos = 8

resultados_clusters = []

for cluster_id in range(k):

    # Ordenar os termos pelo peso no centróide
    indices_ordenados = np.argsort(
        centroides[cluster_id]
    )[::-1]

    principais_indices = indices_ordenados[:n_termos]

    principais_termos = termos[principais_indices]

    resultados_clusters.append({
        "cluster": cluster_id,
        "tokens_representativos": ", ".join(principais_termos)
    })

# Criar DataFrame
df_tokens_clusters = pd.DataFrame(resultados_clusters)

baixar_tabela(df_tokens_clusters, "11_termos_representativos.csv")

df_tokens_clusters

Tabela salva em: tabelas_exportadas/11_termos_representativos.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,cluster,tokens_representativos
0,0,"ancelotti, seleção, convocação, otávio, carlo,..."
1,1,"corinthians, flamengo, estudiantes, gol, jogo,..."
2,2,"cruzeiro, santo, cebolinha, coutinho, peixe, n..."
3,3,"boca, paulo, juniors, tricolor, luciano, doriv..."
4,4,"viña, matías, cruzeiro, river, plate, lateral,..."
5,5,"clube, milhão, futebol, ano, profissional, fla..."
6,6,"botafogo, grêmio, tite, renato, técnico, bellã..."
7,7,"atlético, galo, santo, mrv, jogo, arena, domín..."
8,8,"santo, arão, luan, gabigol, peres, escobar, ro..."
9,9,"fluminense, platense, hulk, tricolor, marcão, ..."


In [ ]:
# Exemplos de manchetes de cada cluster, para ajudar na interpretação
# (limitamos a algumas notícias por cluster para manter a tabela legível)
n_exemplos = 5


def titulos_do_cluster(cluster_id, limite=n_exemplos):
    titulos_cluster = df.loc[
        df["cluster"] == cluster_id,
        "titulo"
    ].tolist()

    amostra = titulos_cluster[:limite]
    texto = "; ".join(amostra)

    if len(titulos_cluster) > limite:
        texto += f" (+{len(titulos_cluster) - limite} outras)"

    return texto


resumo_por_cluster = []

for cluster_id in range(k):
    n_docs_cluster = int((df["cluster"] == cluster_id).sum())

    resumo_por_cluster.append({
        "cluster": cluster_id,
        "n_noticias": n_docs_cluster,
        "exemplos_de_titulos": titulos_do_cluster(cluster_id)
    })

df_resumo_clusters = pd.DataFrame(resumo_por_cluster)

# Juntar com os termos representativos
df_interpretacao_clusters = df_tokens_clusters.merge(
    df_resumo_clusters,
    on="cluster"
)

baixar_tabela(df_interpretacao_clusters, "12_interpretacao_clusters.csv")

df_interpretacao_clusters

Tabela salva em: tabelas_exportadas/12_interpretacao_clusters.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,cluster,tokens_representativos,n_noticias,exemplos_de_titulos
0,0,"ancelotti, seleção, convocação, otávio, carlo,...",26,"Goleiro do mês! Confira números de Otávio, do ..."
1,1,"corinthians, flamengo, estudiantes, gol, jogo,...",73,Jardim define escalação do Flamengo contra o D...
2,2,"cruzeiro, santo, cebolinha, coutinho, peixe, n...",38,Cruzeiro pode ter retorno importante contra o ...
3,3,"boca, paulo, juniors, tricolor, luciano, doriv...",32,"Conmebol denuncia São Paulo, que corre risco d..."
4,4,"viña, matías, cruzeiro, river, plate, lateral,...",9,Números de Viña em comparação com outros later...
5,5,"clube, milhão, futebol, ano, profissional, fla...",95,'Fim do futebol': clubes da Série A se manifes...
6,6,"botafogo, grêmio, tite, renato, técnico, bellã...",37,"Apresentado pelo Botafogo, Tite avalia trabalh..."
7,7,"atlético, galo, santo, mrv, jogo, arena, domín...",41,Neymar se pronuncia após queda do Santos na Su...
8,8,"santo, arão, luan, gabigol, peres, escobar, ro...",14,Santos está escalado para buscar a vaga na Sul...
9,9,"fluminense, platense, hulk, tricolor, marcão, ...",28,Fluminense conhece adversário na semifinal da ...


## Coesão interna dos clusters

Vamos avaliar o quanto as notícias de cada cluster são semelhantes entre si, calculando a **similaridade média entre todos os pares de textos pertencentes ao mesmo cluster**.

- valores mais próximos de **1** indicam clusters mais coesos;
- valores mais baixos indicam maior diversidade interna;
- clusters com poucas notícias podem apresentar coesão instável (poucos pares para calcular a média).

Como o agrupamento foi construído com TF-IDF, usamos a similaridade do cosseno calculada a partir dessa mesma representação.

In [ ]:
import numpy as np

resultados_coesao = []

for cluster_id in sorted(df["cluster"].unique()):

    # Índices dos documentos pertencentes ao cluster
    indices = df.index[df["cluster"] == cluster_id].tolist()

    # Extrair a submatriz de similaridade
    submatriz = similaridade_tfidf[np.ix_(indices, indices)]

    # Selecionar apenas a parte superior da matriz,
    # excluindo a diagonal principal
    valores_pares = submatriz[
        np.triu_indices_from(submatriz, k=1)
    ]

    # Calcular a similaridade média (não definida para clusters com 1 documento)
    coesao_media = valores_pares.mean() if len(valores_pares) > 0 else float("nan")

    resultados_coesao.append({
        "cluster": cluster_id,
        "n_documentos": len(indices),
        "coesao_media": coesao_media
    })


df_coesao = pd.DataFrame(resultados_coesao)

df_coesao

,cluster,n_documentos,coesao_media
0,0,26,0.258197
1,1,73,0.179309
2,2,38,0.214611
3,3,32,0.263483
4,4,9,0.600807
5,5,95,0.059548
6,6,37,0.175603
7,7,41,0.197579
8,8,14,0.386143
9,9,28,0.301096


In [ ]:
df_coesao_completo = df_coesao.merge(
    df_resumo_clusters[["cluster", "exemplos_de_titulos"]],
    on="cluster"
)

df_coesao_completo = df_coesao_completo.sort_values(
    "coesao_media",
    ascending=False
)

baixar_tabela(df_coesao_completo, "13_coesao_clusters.csv")

df_coesao_completo

Tabela salva em: tabelas_exportadas/13_coesao_clusters.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,cluster,n_documentos,coesao_media,exemplos_de_titulos
4,4,9,0.600807,Números de Viña em comparação com outros later...
8,8,14,0.386143,Santos está escalado para buscar a vaga na Sul...
9,9,28,0.301096,Fluminense conhece adversário na semifinal da ...
3,3,32,0.263483,"Conmebol denuncia São Paulo, que corre risco d..."
0,0,26,0.258197,"Goleiro do mês! Confira números de Otávio, do ..."
2,2,38,0.214611,Cruzeiro pode ter retorno importante contra o ...
7,7,41,0.197579,Neymar se pronuncia após queda do Santos na Su...
10,10,52,0.194083,Provocações de Deyverson; relembre momentos do...
11,11,55,0.188370,Vasco trabalha para renovar contrato de dois m...
1,1,73,0.179309,Jardim define escalação do Flamengo contra o D...


## Distância entre os clusters

Além de avaliar se as notícias de um cluster são parecidas entre si, também podemos avaliar o quanto os diferentes clusters estão separados, comparando os **centróides** produzidos pelo K-Means.

Usaremos a **distância do cosseno**:

- valores próximos de **0** indicam clusters semelhantes;
- valores maiores indicam clusters mais diferentes entre si.

A distância é calculada como:

**distância = 1 − similaridade do cosseno**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Similaridade entre os centróides
similaridade_centroides = cosine_similarity(
    kmeans.cluster_centers_
)

# Converter similaridade em distância
distancia_centroides = 1 - similaridade_centroides

# Criar DataFrame
df_distancia_clusters = pd.DataFrame(
    distancia_centroides,
    index=[f"Cluster {i}" for i in range(k)],
    columns=[f"Cluster {i}" for i in range(k)]
).round(3)

baixar_tabela(df_distancia_clusters, "14_distancia_entre_clusters.csv", index=True)

df_distancia_clusters

Tabela salva em: tabelas_exportadas/14_distancia_entre_clusters.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5,Cluster 6,Cluster 7,Cluster 8,Cluster 9,Cluster 10,Cluster 11
Cluster 0,0.000,0.696,0.662,0.803,0.829,0.598,0.714,0.733,0.799,0.801,0.766,0.738
Cluster 1,0.696,0.000,0.619,0.666,0.806,0.509,0.654,0.595,0.741,0.654,0.561,0.678
Cluster 2,0.662,0.619,0.000,0.711,0.718,0.581,0.636,0.528,0.494,0.744,0.664,0.635
Cluster 3,0.803,0.666,0.711,-0.000,0.886,0.615,0.726,0.653,0.788,0.681,0.596,0.685
Cluster 4,0.829,0.806,0.718,0.886,0.000,0.708,0.833,0.853,0.871,0.892,0.861,0.820
Cluster 5,0.598,0.509,0.581,0.615,0.708,-0.000,0.578,0.610,0.776,0.672,0.566,0.527
Cluster 6,0.714,0.654,0.636,0.726,0.833,0.578,-0.000,0.666,0.769,0.765,0.653,0.656
Cluster 7,0.733,0.595,0.528,0.653,0.853,0.610,0.666,-0.000,0.532,0.612,0.619,0.677
Cluster 8,0.799,0.741,0.494,0.788,0.871,0.776,0.769,0.532,-0.000,0.818,0.737,0.797
Cluster 9,0.801,0.654,0.744,0.681,0.892,0.672,0.765,0.612,0.818,0.000,0.639,0.748


In [ ]:
resultados_distancias = []

for i in range(k):
    for j in range(i + 1, k):

        resultados_distancias.append({
            "cluster_1": i,
            "cluster_2": j,
            "distancia": distancia_centroides[i, j]
        })

df_pares_clusters = pd.DataFrame(
    resultados_distancias
).sort_values("distancia")

baixar_tabela(df_pares_clusters, "15_pares_clusters_distancia.csv")

df_pares_clusters

Tabela salva em: tabelas_exportadas/15_pares_clusters_distancia.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,cluster_1,cluster_2,distancia
26,2,8,0.493597
14,1,5,0.508767
50,5,11,0.527323
25,2,7,0.528332
56,7,8,0.531902
...,...,...,...
40,4,7,0.852663
43,4,10,0.860887
41,4,8,0.870888
30,3,4,0.886118


## Clusters mais próximos e mais distantes

A matriz de distância permite identificar dois casos especialmente interessantes:

- o par de clusters com **menor distância**, isto é, os grupos mais semelhantes entre si;
- o par com **maior distância**, isto é, os grupos mais diferentes.

Vamos identificar esses pares e listar exemplos de manchetes de cada cluster envolvido.

In [ ]:
# Identificar menor e maior distância entre clusters
par_mais_proximo = df_pares_clusters.loc[
    df_pares_clusters["distancia"].idxmin()
]

par_mais_distante = df_pares_clusters.loc[
    df_pares_clusters["distancia"].idxmax()
]

# Clusters mais próximos
c1_prox = int(par_mais_proximo["cluster_1"])
c2_prox = int(par_mais_proximo["cluster_2"])

# Clusters mais distantes
c1_dist = int(par_mais_distante["cluster_1"])
c2_dist = int(par_mais_distante["cluster_2"])


print("CLUSTERS MAIS PRÓXIMOS")
print("----------------------")
print(
    f"Cluster {c1_prox} × Cluster {c2_prox}"
)
print(
    f"Distância: {par_mais_proximo['distancia']:.3f}"
)
print(
    f"Cluster {c1_prox}: {titulos_do_cluster(c1_prox)}"
)
print(
    f"Cluster {c2_prox}: {titulos_do_cluster(c2_prox)}"
)


print("\nCLUSTERS MAIS DISTANTES")
print("-----------------------")
print(
    f"Cluster {c1_dist} × Cluster {c2_dist}"
)
print(
    f"Distância: {par_mais_distante['distancia']:.3f}"
)
print(
    f"Cluster {c1_dist}: {titulos_do_cluster(c1_dist)}"
)
print(
    f"Cluster {c2_dist}: {titulos_do_cluster(c2_dist)}"
)

df_clusters_extremos = pd.DataFrame([
    {
        "tipo": "mais próximos",
        "cluster_1": c1_prox,
        "cluster_2": c2_prox,
        "distancia": par_mais_proximo["distancia"]
    },
    {
        "tipo": "mais distantes",
        "cluster_1": c1_dist,
        "cluster_2": c2_dist,
        "distancia": par_mais_distante["distancia"]
    }
])

baixar_tabela(df_clusters_extremos, "16_clusters_mais_proximos_e_distantes.csv")

CLUSTERS MAIS PRÓXIMOS
----------------------
Cluster 2 × Cluster 8
Distância: 0.494
Cluster 2: Cruzeiro pode ter retorno importante contra o Vitória; Santos: Após estrear, Cebolinha ficará quase um mês sem jogar; Dupla do Santos lamenta eliminação na Sul-Americana; Santos: Cebolinha pode jogar a Sul-Americana?; Cruzeiro chega a dez jogos marcando e sofrendo gols (+33 outras)
Cluster 8: Santos está escalado para buscar a vaga na Sul-Americana; Santos busca classificação na Sul-Americana sob alerta de Cuca; Escalação: Santos tem retorno de titular para jogo da Sula; Santos divulga lista de relacionados para enfrentar o Atlético-MG; Cuca explica esquema tático e elogia atuação do Santos (+9 outras)

CLUSTERS MAIS DISTANTES
-----------------------
Cluster 4 × Cluster 9
Distância: 0.892
Cluster 4: Números de Viña em comparação com outros laterais do Cruzeiro; Matías Viña projeta disputa por posição no Cruzeiro; Reforço do Cruzeiro, Viña abre o jogo sobre condição física; Quem paga salário?

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusão

Neste notebook, aplicamos sobre a base real de notícias do nosso projeto a mesma sequência de análise de texto demonstrada em aula com o dataset sintético de animais:

- Reaproveitamos a preparação textual (tokenização, normalização, remoção de *stopwords*, *stemming* e lematização) já feita pelo pipeline de coleta do projeto (`coletar_lance.py` + `preprocessar.py`).
- Criamos representações numéricas com **Bag of Words** e **TF-IDF** a partir dos lemas de cada notícia.
- Calculamos a **similaridade do cosseno** entre as notícias, comparando as representações BoW e TF-IDF.
- Agrupamos as notícias com **K-Means**, escolhendo *k* a partir das seções conhecidas da base **e** validando essa escolha com o **método do cotovelo** e o **coeficiente de silhueta** — diferente do exemplo da aula, em que o número de grupos já era conhecido de antemão.
- Visualizamos os clusters com **PCA em 2D e 3D**.
- Interpretamos os clusters pelos **termos mais representativos** de cada centróide e por exemplos de manchetes.
- Avaliamos a **coesão interna** de cada cluster e a **distância entre centróides**, identificando os grupos mais semelhantes e mais distintos.

Diferente da base sintética de animais, o texto jornalístico traz ruído real (nomes próprios, números, variações de estilo entre repórteres), o que torna a escolha de *k* e a interpretação dos clusters mais desafiadoras — e mais próximas de um cenário real de aplicação dessas técnicas, como sistemas de recomendação de notícias, agrupamento automático por assunto e sumarização de coberturas esportivas.
